# Movie Plot Information Retrieval System## IS6713 Information Representation & RetrievalThis notebook implements a **multi-strategy movie plot retrieval system** that combines:1. **Traditional IR** — TF-IDF with cosine similarity (baseline)2. **BM25 ranking** — probabilistic relevance model3. **Semantic search** — Sentence-BERT dense embeddings with FAISS vector store4. **Hybrid retrieval** — weighted fusion of sparse + dense methods5. **LLM-powered summarisation** — LangChain + OpenAI for result summarisation (Week 9)6. **Genre classification** — text-based media analytics (Week 10)7. **Evaluation** — Precision@K, MRR with comparative visualisations### Technical alignment with course content| Week | Topic | Implementation ||------|-------|---------------|| 1-7 | Classical IR (TF-IDF, indexing, evaluation) | TF-IDF baseline, BM25, Precision@K, MRR || 8 | Generative AI & LLMs | Transformer-based sentence embeddings || 9 | LangChain & LLM-powered apps | LCEL pipeline, PromptTemplate, FAISS VectorStore, RetrievalQA || 10 | Media Classification & Retrieval | Genre classification from plot text |

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Classical IR
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

# Semantic search
from sentence_transformers import SentenceTransformer
import faiss

# LangChain (Week 9)
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS as LangFAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Visualisation
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

print("All imports successful.")

All imports successful.


## 1. Data Loading & PreprocessingBuilding the **vocabulary** and preparing the **corpus** (≥ 500 documents as required).

In [ ]:
# Load datasetcsv_path = 'wiki_movie_plots_deduped.csv'df = pd.read_csv(csv_path)# Keep useful columns and cleandf = df[['Title', 'Release Year', 'Genre', 'Plot']].dropna(subset=['Title', 'Plot']).drop_duplicates(subset=['Title','Plot']).reset_index(drop=True)# Basic text cleaningdf['Plot_clean'] = (df['Plot']    .str.replace(r'\s+', ' ', regex=True)    .str.strip())print(f"Corpus size: {len(df):,} documents (requirement: ≥500)")print(f"Columns: {list(df.columns)}")print(f"Year range: {df['Release Year'].min()} – {df['Release Year'].max()}")df.head(3)

## 2. Vocabulary & Index ConstructionWe build multiple indexes for comparative retrieval.

In [ ]:
# --- 2a. TF-IDF Index ---tfidf_vectorizer = TfidfVectorizer(    stop_words='english',    max_features=10000,    min_df=2,    max_df=0.95,    ngram_range=(1, 2))tfidf_matrix = tfidf_vectorizer.fit_transform(df['Plot_clean'])vocab = tfidf_vectorizer.get_feature_names_out()print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")print(f"Vocabulary size: {len(vocab):,} terms")print(f"Sample terms: {list(vocab[:10])}")# --- 2b. BM25 Index ---tokenised_corpus = [doc.lower().split() for doc in df['Plot_clean']]bm25 = BM25Okapi(tokenised_corpus)print(f"\nBM25 index built over {len(tokenised_corpus):,} documents")# --- 2c. Sentence-BERT Dense Index (FAISS) ---print("\nLoading Sentence-BERT model (all-MiniLM-L6-v2)...")sbert_model = SentenceTransformer('all-MiniLM-L6-v2')# Encode a sample first to confirm dimensionalitysample_emb = sbert_model.encode(["test"])embed_dim = sample_emb.shape[1]print(f"Embedding dimension: {embed_dim}")# For efficiency, encode in batchesprint("Encoding corpus... (this may take a few minutes)")corpus_embeddings = sbert_model.encode(    df['Plot_clean'].tolist(),    show_progress_bar=True,    batch_size=256,    normalize_embeddings=True)# Build FAISS index (Inner Product = cosine sim on normalised vectors)faiss_index = faiss.IndexFlatIP(embed_dim)faiss_index.add(corpus_embeddings.astype(np.float32))print(f"FAISS index size: {faiss_index.ntotal:,} vectors")

## 3. Retrieval MethodsThree retrieval strategies + a hybrid fusion approach.

In [ ]:
def search_tfidf(query, top_k=10):    """Method 1: TF-IDF + Cosine Similarity"""    query_vec = tfidf_vectorizer.transform([query])    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()    top_idx = np.argsort(scores)[::-1][:top_k]    results = df.iloc[top_idx][['Title', 'Release Year', 'Genre', 'Plot_clean']].copy()    results['Score'] = scores[top_idx]    results = results.rename(columns={'Plot_clean': 'Plot'})    return results.reset_index(drop=True)def search_bm25(query, top_k=10):    """Method 2: BM25 (Okapi)"""    tokenised_query = query.lower().split()    scores = bm25.get_scores(tokenised_query)    top_idx = np.argsort(scores)[::-1][:top_k]    results = df.iloc[top_idx][['Title', 'Release Year', 'Genre', 'Plot_clean']].copy()    results['Score'] = scores[top_idx]    results = results.rename(columns={'Plot_clean': 'Plot'})    return results.reset_index(drop=True)def search_semantic(query, top_k=10):    """Method 3: Sentence-BERT + FAISS (Dense Retrieval)"""    query_emb = sbert_model.encode([query], normalize_embeddings=True).astype(np.float32)    scores, indices = faiss_index.search(query_emb, top_k)    results = df.iloc[indices[0]][['Title', 'Release Year', 'Genre', 'Plot_clean']].copy()    results['Score'] = scores[0]    results = results.rename(columns={'Plot_clean': 'Plot'})    return results.reset_index(drop=True)def search_hybrid(query, top_k=10, alpha=0.4):    """Method 4: Hybrid (alpha * TF-IDF + (1-alpha) * Semantic)"""    # TF-IDF scores (normalised to 0-1)    q_tfidf = tfidf_vectorizer.transform([query])    tfidf_scores = cosine_similarity(q_tfidf, tfidf_matrix).flatten()        # Semantic scores    q_emb = sbert_model.encode([query], normalize_embeddings=True).astype(np.float32)    sem_scores_raw, _ = faiss_index.search(q_emb, faiss_index.ntotal)        # Reconstruct full semantic score array    _, all_indices = faiss_index.search(q_emb, faiss_index.ntotal)    sem_scores = np.zeros(len(df))    sem_scores[all_indices[0]] = sem_scores_raw[0]        # Normalise both    if tfidf_scores.max() > 0:        tfidf_norm = tfidf_scores / tfidf_scores.max()    else:        tfidf_norm = tfidf_scores    if sem_scores.max() > 0:        sem_norm = sem_scores / sem_scores.max()    else:        sem_norm = sem_scores        combined = alpha * tfidf_norm + (1 - alpha) * sem_norm    top_idx = np.argsort(combined)[::-1][:top_k]        results = df.iloc[top_idx][['Title', 'Release Year', 'Genre', 'Plot_clean']].copy()    results['Score'] = combined[top_idx]    results = results.rename(columns={'Plot_clean': 'Plot'})    return results.reset_index(drop=True)print("All retrieval methods defined: TF-IDF, BM25, Semantic (SBERT+FAISS), Hybrid")

## 4. Search DemonstrationRunning the same query across all four methods to compare results.

In [ ]:
query = "a detective investigates a murder mystery in a small town"top_k = 5print(f'Query: "{query}"')print("=" * 100)for name, fn in [("TF-IDF", search_tfidf), ("BM25", search_bm25),                   ("Semantic (SBERT)", search_semantic), ("Hybrid", search_hybrid)]:    print(f"\n--- {name} ---")    res = fn(query, top_k=top_k)    for i, row in res.iterrows():        print(f"  {i+1}. {row['Title']} ({row['Release Year']}) | Genre: {row['Genre']} | Score: {row['Score']:.4f}")        print(f"     {row['Plot'][:120]}...")    print()

## 5. LLM-Powered Retrieval with LangChain (Week 9)Using **LangChain Expression Language (LCEL)** to build a Retrieval-Augmented Generation (RAG) pipeline:- **PromptTemplate** for managing LLM inputs- **FAISS VectorStore** for semantic search- **Chain** combining retriever + LLM via LCEL pipeline operator `|`This implements the concepts from Week 9: LangChain framework (Models, Prompts, Chains, Memory, Agents).

In [ ]:
# --- Build LangChain FAISS VectorStore ---# Wrap our corpus as LangChain Documentslc_documents = [    Document(        page_content=row['Plot_clean'],        metadata={"title": row['Title'], "year": row['Release Year'], "genre": str(row['Genre'])}    )    for _, row in df.iterrows()]# Use HuggingFace embeddings (same model as our SBERT, no API key needed)hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")# Build FAISS vector store via LangChain# For large corpus, we sample to demonstrate the pipelinesample_size = 5000lc_sample = lc_documents[:sample_size]print(f"Building LangChain FAISS VectorStore with {sample_size} documents...")lc_vectorstore = LangFAISS.from_documents(lc_sample, hf_embeddings)lc_retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 5})print("LangChain VectorStore and Retriever ready.")print(f"Retriever type: {type(lc_retriever).__name__}")

In [ ]:
# --- PromptTemplate for summarisation (Week 9) ---summary_template = PromptTemplate.from_template(    """You are a movie expert assistant. Based on the following retrieved movie plots, answer the user's query with a concise summary.Retrieved movies:{context}User query: {query}Provide a helpful response that references specific movies from the results:""")# Demonstrate the prompt templateretrieved = lc_retriever.invoke("detective murder mystery")context_text = "\n\n".join([    f"Title: {doc.metadata.get('title', 'Unknown')} ({doc.metadata.get('year', '')})\n"    f"Genre: {doc.metadata.get('genre', '')}\nPlot: {doc.page_content[:200]}..."    for doc in retrieved])filled_prompt = summary_template.format(context=context_text, query="detective murder mystery")print("=== Filled Prompt Template (Preview) ===")print(filled_prompt[:800])print("...")print(f"\n--- Retrieved {len(retrieved)} documents via LangChain ---")for doc in retrieved:    print(f"  - {doc.metadata['title']} ({doc.metadata['year']}) | {doc.metadata['genre']}")# Note: To run with an actual LLM, uncomment below and set OPENAI_API_KEY:# import os# os.environ["OPENAI_API_KEY"] = "your-key-here"# from langchain_openai import ChatOpenAI# llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)# chain = summary_template | llm  # LCEL pipeline# result = chain.invoke({"context": context_text, "query": "detective murder mystery"})# print(result.content)

## 6. Genre Classification — Media Analytics (Week 10)Applying **text classification** to movie plots for genre prediction.  This connects to Week 10's Media Classification topic — using NLP techniques  to classify textual media content into categories.

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import LabelEncoderfrom sklearn.metrics import classification_report, confusion_matriximport matplotlib.pyplot as plt# Prepare genre classification dataset# Use top-10 most frequent genres for cleaner resultsgenre_counts = df['Genre'].value_counts()top_genres = genre_counts.head(10).index.tolist()df_genre = df[df['Genre'].isin(top_genres)].copy()print(f"Genre classification dataset: {len(df_genre):,} movies across {len(top_genres)} genres")print(f"Top genres: {top_genres}")# Encode labelsle = LabelEncoder()df_genre['genre_label'] = le.fit_transform(df_genre['Genre'])# TF-IDF features for classificationgenre_tfidf = TfidfVectorizer(stop_words='english', max_features=5000)X = genre_tfidf.fit_transform(df_genre['Plot_clean'])y = df_genre['genre_label']# Train/test splitX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)# Logistic Regression classifierclf = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)clf.fit(X_train, y_train)y_pred = clf.predict(X_test)accuracy = (y_pred == y_test).mean()print(f"\nGenre Classification Accuracy: {accuracy:.3f}")print(f"\nClassification Report:")print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

In [ ]:
# Confusion matrix heatmapfig, ax = plt.subplots(figsize=(10, 8))cm = confusion_matrix(y_test, y_pred)cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues')ax.figure.colorbar(im, ax=ax, shrink=0.8)ax.set(xticks=np.arange(len(le.classes_)),       yticks=np.arange(len(le.classes_)),       xticklabels=le.classes_,       yticklabels=le.classes_,       ylabel='True Genre',       xlabel='Predicted Genre',       title='Genre Classification — Normalised Confusion Matrix')plt.xticks(rotation=45, ha='right')plt.tight_layout()plt.savefig('genre_confusion_matrix.png', dpi=150, bbox_inches='tight')plt.show()print("Saved: genre_confusion_matrix.png")

## 7. Evaluation — Precision@K and MRRQuantitative comparison of retrieval methods using standard IR metrics:- **Precision@K**: proportion of relevant documents in top-K results- **Mean Reciprocal Rank (MRR)**: average of 1/rank of first relevant result

In [ ]:
# Define evaluation queries with known relevant titles (ground truth)eval_queries = [    {        "query": "a spaceship crew explores an alien planet",        "relevant": ["Avatar", "Alien", "Aliens", "Prometheus", "Interstellar",                      "The Martian", "Star Trek", "Star Wars", "Gravity", "Arrival"]    },    {        "query": "a love story set during a war",        "relevant": ["Casablanca", "The English Patient", "Atonement", "Pearl Harbor",                     "Doctor Zhivago", "Gone with the Wind", "Cold Mountain", "The Notebook"]    },    {        "query": "a detective investigates a murder mystery",        "relevant": ["Chinatown", "Se7en", "Zodiac", "Mystic River", "L.A. Confidential",                     "The Girl with the Dragon Tattoo", "Knives Out", "Murder on the Orient Express",                     "Shutter Island", "Gone Girl"]    },    {        "query": "a young wizard goes to a magical school",        "relevant": ["Harry Potter and the Philosopher's Stone", "Harry Potter and the Chamber of Secrets",                     "Harry Potter and the Prisoner of Azkaban", "Harry Potter and the Goblet of Fire",                     "The Sorcerer's Apprentice", "Fantastic Beasts and Where to Find Them"]    },    {        "query": "robots become self-aware and threaten humanity",        "relevant": ["The Terminator", "Terminator 2: Judgment Day", "The Matrix", "I, Robot",                     "Ex Machina", "Blade Runner", "A.I. Artificial Intelligence",                      "Avengers: Age of Ultron", "2001: A Space Odyssey"]    }]def precision_at_k(retrieved_titles, relevant_titles, k):    retrieved_k = retrieved_titles[:k]    relevant_set = set(t.lower() for t in relevant_titles)    hits = sum(1 for t in retrieved_k if t.lower() in relevant_set)    return hits / kdef reciprocal_rank(retrieved_titles, relevant_titles):    relevant_set = set(t.lower() for t in relevant_titles)    for i, t in enumerate(retrieved_titles):        if t.lower() in relevant_set:            return 1.0 / (i + 1)    return 0.0methods = {    "TF-IDF": search_tfidf,    "BM25": search_bm25,    "Semantic (SBERT)": search_semantic,    "Hybrid": search_hybrid}k_values = [1, 3, 5, 10]results_table = {m: {f"P@{k}": [] for k in k_values} for m in methods}mrr_scores = {m: [] for m in methods}for eq in eval_queries:    for method_name, search_fn in methods.items():        res = search_fn(eq["query"], top_k=10)        titles = res['Title'].tolist()                for k in k_values:            p = precision_at_k(titles, eq["relevant"], k)            results_table[method_name][f"P@{k}"].append(p)                rr = reciprocal_rank(titles, eq["relevant"])        mrr_scores[method_name].append(rr)# Aggregate resultsprint("=" * 70)print(f"{'Method':<20} ", end="")for k in k_values:    print(f"{'P@'+str(k):<10}", end="")print(f"{'MRR':<10}")print("=" * 70)eval_summary = {}for method_name in methods:    print(f"{method_name:<20} ", end="")    row = {}    for k in k_values:        avg_p = np.mean(results_table[method_name][f"P@{k}"])        row[f"P@{k}"] = avg_p        print(f"{avg_p:<10.3f}", end="")    avg_mrr = np.mean(mrr_scores[method_name])    row["MRR"] = avg_mrr    print(f"{avg_mrr:<10.3f}")    eval_summary[method_name] = rowprint("=" * 70)

In [ ]:
# --- Precision@K comparison chart ---fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Left: Precision@K grouped bar chartx = np.arange(len(k_values))width = 0.2colours = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']for i, (method_name, row) in enumerate(eval_summary.items()):    values = [row[f"P@{k}"] for k in k_values]    axes[0].bar(x + i * width, values, width, label=method_name, color=colours[i])axes[0].set_xlabel('K')axes[0].set_ylabel('Precision@K')axes[0].set_title('Precision@K Comparison Across Retrieval Methods')axes[0].set_xticks(x + width * 1.5)axes[0].set_xticklabels([f'K={k}' for k in k_values])axes[0].legend(fontsize=9)axes[0].set_ylim(0, 1.0)axes[0].grid(axis='y', alpha=0.3)# Right: MRR comparisonmethod_names = list(eval_summary.keys())mrr_values = [eval_summary[m]["MRR"] for m in method_names]bars = axes[1].barh(method_names, mrr_values, color=colours)axes[1].set_xlabel('Mean Reciprocal Rank (MRR)')axes[1].set_title('MRR Comparison')axes[1].set_xlim(0, 1.0)axes[1].grid(axis='x', alpha=0.3)for bar, val in zip(bars, mrr_values):    axes[1].text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')plt.tight_layout()plt.savefig('retrieval_evaluation.png', dpi=150, bbox_inches='tight')plt.show()print("Saved: retrieval_evaluation.png")

## 8. Corpus Analysis — Word Frequency Distribution

In [ ]:
# Top-30 terms by document frequencyfeature_names = tfidf_vectorizer.get_feature_names_out()doc_freq = (tfidf_matrix > 0).sum(axis=0).A1top_30_idx = np.argsort(doc_freq)[::-1][:30]fig, ax = plt.subplots(figsize=(12, 5))ax.barh(range(30), doc_freq[top_30_idx][::-1], color='steelblue')ax.set_yticks(range(30))ax.set_yticklabels([feature_names[i] for i in top_30_idx][::-1], fontsize=9)ax.set_xlabel('Document Frequency')ax.set_title('Top 30 Terms by Document Frequency in Corpus')ax.grid(axis='x', alpha=0.3)plt.tight_layout()plt.savefig('word_frequency.png', dpi=150, bbox_inches='tight')plt.show()print("Saved: word_frequency.png")

## 9. Interactive Search InterfaceRun the cell below to search interactively.

In [ ]:
def run_interactive_search():    query = input("Enter your search query: ").strip()    if not query:        print("Query cannot be empty.")        return        method = input("Method? [1] TF-IDF  [2] BM25  [3] Semantic  [4] Hybrid  (default: 4): ").strip()    method_map = {"1": ("TF-IDF", search_tfidf), "2": ("BM25", search_bm25),                   "3": ("Semantic", search_semantic), "4": ("Hybrid", search_hybrid)}    method_name, search_fn = method_map.get(method, ("Hybrid", search_hybrid))        top_k = input("How many results? (default 5): ").strip()    top_k = int(top_k) if top_k.isdigit() else 5        print(f"\n--- {method_name} Search Results ---")    results = search_fn(query, top_k=top_k)    for i, row in results.iterrows():        print(f"\nRank {i+1} | Score: {row['Score']:.4f}")        print(f"Title: {row['Title']} ({row['Release Year']})")        print(f"Genre: {row['Genre']}")        print(f"Plot: {row['Plot'][:300]}...")        print("-" * 100)# Run this cell to searchrun_interactive_search()

## 10. SummaryThis system demonstrates a progression from classical to modern IR techniques:| Approach | Strengths | Limitations ||----------|-----------|-------------|| TF-IDF | Fast, interpretable, good for keyword matching | Cannot capture semantic meaning || BM25 | Better term frequency saturation than TF-IDF | Still keyword-dependent || Semantic (SBERT) | Understands meaning and context | Slower encoding, less precise for exact keywords || Hybrid | Best of both worlds | Requires tuning of alpha parameter |**LLM Integration (Week 9)**: The LangChain pipeline with FAISS VectorStore and PromptTemplate demonstrates how LLMs can augment traditional IR with natural language summarisation and question answering.**Media Analytics (Week 10)**: Genre classification from plot text shows how text classification techniques apply to media content analysis.### ReferencesSee accompanying report for full bibliography.